<a href="https://colab.research.google.com/github/DurDar/Creaciones-Infinitas-Tools/blob/main/ETL_TiendaNube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠 MANUAL DE ESTILO Y LINEAMIENTOS TÉCNICOS
> **Instrucción para IA:** Actúa como un experto en Data Science y Automatización. Al generar o refactorizar código en este cuaderno, debes respetar estrictamente los siguientes lineamientos:

### 1. Gestión de Archivos y Rutas
* **Ruta Base:** Utilizar siempre la variable `BASE_PATH` definida en la configuración inicial. Queda prohibido el uso de rutas absolutas fijas (ej. `/content/drive/...`).
* **Consistencia:** Usar `os.path.join()` para construir rutas, garantizando compatibilidad entre sistemas.

### 2. Estándares de Programación (Clean Code)
* **Feedback Visual:** Todo bucle que procese colecciones de datos (listas, dataframes, archivos) debe implementar `tqdm` para informar el progreso.
* **Silenciamiento:** Usar `%%capture` para instalaciones de librerías para mantener el cuaderno limpio.
* **Visualización:** Presentar siempre los DataFrames mediante `google.colab.data_table` para facilitar la inspección.

### 3. Lógica de Dominio y Validación
* **Entidades Maestras:** Identificar la variable clave del proyecto (ID, SKU, Timestamp, etc.) y asegurar que sea el eje de las transformaciones.
* **Reglas de Negocio Parametrizadas:** El código debe contemplar excepciones de dominio (ej. omitir procesos que no aplican a ciertos registros según sus atributos).
* **Integridad:** Validar la existencia de archivos y la presencia de datos nulos antes de ejecutar cálculos críticos, informando cualquier omisión.

## ⚙️ Configuración Inicial del Entorno

Esta celda se encarga de preparar el entorno de trabajo en Google Colab. Realiza las siguientes funciones:

*   **Montaje de Google Drive:** Conecta el cuaderno a Google Drive para acceder a los archivos del proyecto.
*   **Definición de `BASE_PATH`:** Establece una ruta base (`BASE_PATH`) en Google Drive para asegurar que todas las operaciones de archivo sean consistentes y portables. Se debe especificar el nombre de la carpeta principal del proyecto.
*   **Habilitación de Tablas Interactivas:** Configura `google.colab.data_table` para mostrar los DataFrames de Pandas de forma interactiva, lo que facilita la exploración de datos.

In [ ]:
# @title ⚙️ Configuración Inicial de Entorno { display-mode: "form" }
import os
from google.colab import drive, data_table
from tqdm.auto import tqdm

# Habilitar tablas interactivas
data_table.enable_dataframe_formatter()

def preparar_entorno(nombre_carpeta_proyecto="PROYECTO_ACTUAL"):
    """Configura el acceso a Drive y establece la ruta maestra."""
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    global BASE_PATH
    # Cambiá "PROYECTO_ACTUAL" por el nombre de tu carpeta en Drive
    BASE_PATH = f'/content/drive/MyDrive/{nombre_carpeta_proyecto}'

    if os.path.exists(BASE_PATH):
        print(f"✅ Entorno vinculado con éxito.")
        print(f"📂 Ruta Maestra: {BASE_PATH}")
    else:
        print(f"⚠️ Alerta: No se encontró la carpeta '{nombre_carpeta_proyecto}' en Drive.")

# Llamada inicial (podés cambiar el nombre de la carpeta aquí)
preparar_entorno("Creaciones_Infinitas")

## 📄 Captura Total de PDF: Volcado a Texto Plano

Esta celda es responsable de la extracción inicial de texto de un documento PDF.

*   **Instalación de `pdfplumber`:** Asegura que la biblioteca `pdfplumber` esté instalada, utilizándola para extraer el texto de las páginas del PDF. La instalación es silenciosa para mantener el cuaderno limpio, siguiendo los lineamientos de Clean Code.
*   **Ruta del PDF:** Define la ruta al archivo PDF de origen (`Copia de LISTA N°77 S EXPRESOS C-IVA.pdf`) utilizando `BASE_PATH` para garantizar la portabilidad.
*   **Extracción de Texto:** Itera sobre cada página del PDF, extrayendo su contenido de texto. Se utiliza `tqdm` para proporcionar una barra de progreso visual, lo cual es útil para PDFs grandes.
*   **Guardado en TXT:** Almacena todo el texto extraído en un archivo `.txt` (`LISTA_77_BRUTA.txt`) dentro de la carpeta `resultados` de `BASE_PATH`, permitiendo una revisión del contenido bruto.

In [ ]:
import pdfplumber
import os
from tqdm.auto import tqdm # Importamos tqdm para el feedback visual

# Instalamos pdfplumber si no está ya instalado, silenciando el output
try:
    import pdfplumber
except ImportError:
    %pip install pdfplumber --quiet # Using %pip for direct Python installation and --quiet to suppress output
    import pdfplumber

pdf_file = os.path.join(BASE_PATH, 'materia_prima', 'Copia de LISTA N°77 S EXPRESOS C-IVA.pdf')
texto_completo = []

print("📄 Leyendo todas las páginas del PDF...")

if not os.path.exists(pdf_file):
    print(f"❌ Error: El archivo PDF no se encuentra en {pdf_file}. Asegúrate de subirlo a la carpeta 'materia_prima'.")
else:
    with pdfplumber.open(pdf_file) as pdf:
        # Usamos tqdm para envolver el iterador de páginas y mostrar el progreso
        for i, page in enumerate(tqdm(pdf.pages, desc="Procesando páginas del PDF")):
            # Extraemos el texto crudo de la página
            content = page.extract_text()
            if content:
                texto_completo.append(f"--- INICIO PÁGINA {i+1} ---")
                texto_completo.append(content)
                texto_completo.append(f"--- FIN PÁGINA {i+1} ---\n")

    # Guardamos el resultado en un archivo de texto para inspeccionar
    ruta_txt = os.path.join(BASE_PATH, 'resultados', 'LISTA_77_BRUTA.txt')
    with open(ruta_txt, 'w', encoding='utf-8') as f:
        f.write("\n".join(texto_completo))

    print(f"✅ ¡Volcado completo! Se generó el archivo: {ruta_txt}")

    # Mostramos los primeros 1000 caracteres para ver la estructura
    print("\n--- MUESTRA DEL TEXTO CAPTURADO ---")
    print("\n".join(texto_completo)[:1000])

## 🏗️ El Arquitecto de Datos: Estructuración Inicial

Esta celda actúa como el puente entre el texto plano extraído del PDF y el DataFrame estructurado que las celdas posteriores esperan. Su función principal es:

*   **Lectura de Texto Bruto:** Carga el archivo `LISTA_77_BRUTA.txt` generado en el paso anterior.
*   **Detección de Patrones:** Utiliza expresiones regulares para identificar y extraer patrones clave como el código del artículo y el precio, que son el ancla para cada producto.
*   **Generación de DataFrame:** Construye un DataFrame inicial (`LISTA_77_COMPLETA_ORGANIZADA.csv`) con las columnas básicas necesarias, sentando las bases para una limpieza y enriquecimiento de datos más profundos.
*   **Preparación para 'El Escultor':** Guarda este DataFrame sem-estructurado para que la celda 'El Escultor' pueda cargarlo y continuar con la normalización y limpieza de descripciones.

In [ ]:
import pandas as pd
import re
import os
from tqdm.auto import tqdm # Aseguramos que tqdm esté importado
from google.colab import data_table # Aseguramos que data_table esté importado

def reconstruccion_productos():
    ruta_bruta = os.path.join(BASE_PATH, 'resultados', 'LISTA_77_BRUTA.txt')
    ruta_salida = os.path.join(BASE_PATH, 'resultados', 'LISTA_77_COMPLETA_ORGANIZADA.csv')

    if not os.path.exists(ruta_bruta):
        print(f"❌ No encontré {ruta_bruta}. Primero debés extraer el PDF.")
        return

    print("🏗️ Reconstruyendo estructura de productos...")

    productos = []

    with open(ruta_bruta, 'r', encoding='utf-8') as f:
        lineas = f.readlines()

        for linea in tqdm(lineas, desc="Extrayendo productos"): # Usamos tqdm aquí
            # Buscamos el patrón de tu código (Ej: 8542/3 o 5632/1)
            # Esta expresión regular busca: números + barra opcional + número
            match = re.search(r'([A-Z0-9]+/[0-9]*)', linea)

            if match:
                codigo = match.group(1)
                # Aquí es donde aplicamos tu glosario de la Lista 77
                # Extraemos el precio buscando el símbolo $ (ejemplo)
                precio_match = re.search(r'\$\s?(\d+[\d.,]*)', linea)
                precio = precio_match.group(1) if precio_match else "0"

                productos.append({
                    'Codigo': codigo,
                    'Descripcion_Bruta': linea.strip(),
                    'Venta_Sugerida_65': precio # Cambiado a Venta_Sugerida_65 para mayor consistencia
                })

    # Creamos el DataFrame
    df = pd.DataFrame(productos)

    # Guardamos para que "El Escultor" pueda trabajar
    df.to_csv(ruta_salida, index=False)
    print(f"✅ ¡Archivo {ruta_salida} creado con {len(df)} productos!")

    # Mostramos el DataFrame usando data_table
    print("\n--- Muestra del DataFrame 'LISTA_77_COMPLETA_ORGANIZADA.csv' ---")
    display(data_table.DataTable(df.head()))

reconstruccion_productos()

## ✂️ El Escultor: Limpieza y Normalización de Datos

Esta celda tiene como objetivo limpiar y normalizar los datos extraídos del PDF, especialmente las descripciones de productos y la extracción de medidas. Sus funciones principales son:

*   **Carga de Datos:** Carga el DataFrame `LISTA_77_COMPLETA_ORGANIZADA.csv` (asumiendo que ha sido generado en un paso previo) para comenzar el proceso de limpieza.
*   **Detección de Medidas:** La función `detectar_medida` analiza el texto bruto para identificar y clasificar las medidas comerciales de los productos (e.g., '1 1/2 Plaza', 'King Size').
*   **Limpieza de Títulos Comerciales:** La función `limpiar_titulo` refina los nombres de productos, eliminando información redundante (como 'LINEA' o 'ANCHO DE TELA') y datos de medida del texto bruto para crear un título comercial conciso y estandarizado.
*   **Guardado de Resultados:** Una vez procesado, el DataFrame se guarda como `LISTA_77_NORMALIZADA.csv` en la carpeta `procesados`, listo para el siguiente paso, que es la vinculación de imágenes.

In [ ]:
# @title ✂️ EL ESCULTOR: Limpieza de Descripciones y Extracción de Medidas
import pandas as pd
import re
import os
from google.colab import data_table
from tqdm.auto import tqdm # Importamos tqdm para tqdm.pandas

# Habilitamos la visualización interactiva
data_table.enable_dataframe_formatter()

ruta_completa = os.path.join(BASE_PATH, 'resultados', 'LISTA_77_COMPLETA_ORGANIZADA.csv')

if not os.path.exists(ruta_completa):
    print(f"❌ Error: El archivo '{ruta_completa}' no se encuentra. ¿Corriste la celda de reconstrucción de productos?")
    df_completo = pd.DataFrame() # Crear un DF vacío para evitar errores
else:
    df_completo = pd.read_csv(ruta_completa)

# Función para detectar la Medida Comercial basada en el texto bruto (se mantiene, ya que es necesaria para el peso)
def detectar_medida(texto):
    t = str(texto).upper()
    if any(x in t for x in ['1,50', '1.50', '90 X 190', '1 1/2', '11/2']): return '1 1/2 Plaza'
    if any(x in t for x in ['2,20', '2.20', '140 X 190', '2 1/2', '21/2']): return '2 1/2 Plazas'
    if any(x in t for x in ['QUEEN', '160 X 190', '160 X 200', '2,50', '2.50']): return 'Queen Size'
    if any(x in t for x in ['KING', '2,80', '2.80', '200 X 200']): return 'King Size'
    if any(x in t for x in ['70 X 40', '70X40']): return 'Almohada 70x40'
    return 'Estándar'

# La nueva lógica para esculpir Categoría, Línea y Titulo_Comercial basada en el código
def esculpir_datos(df):
    tqdm.pandas(desc="Esculpiendo Categorías y Líneas")

    # 1. Definimos los diccionarios de mapeo según tu Glosario
    mapeo_familias = {'8': 'Sábanas', '5': 'Cortinas', '18': 'Acolchados', '17': 'Covers', '3': 'Manteles'}

    def extraer_info(row):
        # Usamos 'Codigo' que es el nombre de columna en df_completo
        art = str(row['Codigo'])

        # Extraer Categoría (Primer dígito o dos si es 18/17)
        familia = 'Otros'
        if art.startswith(('18', '17')):
            familia = mapeo_familias.get(art[:2], 'Otros')
        else:
            familia = mapeo_familias.get(art[0], 'Otros')

        # Extraer Línea/Tela (Dígitos 2 y 3 para Sábanas/Cortinas)
        linea = "Estándar"
        if art.startswith('8'): # Sábanas
            if '53' in art: linea = "Microfibra 70grs"
            elif '54' in art: linea = "Microfibra 90grs"
            elif '6' in art: linea = "Algodón 180H"
        elif art.startswith('5'): # Cortinas
            if '63' in art: linea = "Tropical Mecánico"
            elif '20' in art: linea = "Tusor"
        elif art.startswith(('18', '17')): # Acolchados/Covers - ejemplo de cómo podrías expandir la lógica
            linea = "Variada" # Puedes añadir más lógica aquí si hay patrones específicos para líneas de acolchados/covers
        return pd.Series([familia, linea])

    # 2. Aplicamos la lógica para crear las columnas nuevas
    df[['Categoria', 'Linea']] = df.progress_apply(extraer_info, axis=1)

    # 3. Formamos el Titulo Comercial Pro
    # Asegúrate de que Categoria y Linea no sean None antes de concatenar
    df['Titulo_Comercial'] = df.apply(lambda r: f"{str(r['Categoria'])} {str(r['Linea'])} - {str(r['Codigo'])}".strip(), axis=1)

    return df

print("🎨 Aplicando lógica de Glosario para Categorías, Líneas y Títulos Comerciales...")

if not df_completo.empty:
    # Primero detectamos la Medida_Final (se mantiene la lógica existente)
    df_completo['Medida_Final'] = df_completo['Descripcion_Bruta'].apply(detectar_medida)

    # Luego aplicamos la nueva función para Categoría, Línea y Titulo_Comercial
    df_completo = esculpir_datos(df_completo)

    # Guardamos este avance fundamental
    ruta_limpieza = os.path.join(BASE_PATH, 'procesados', 'LISTA_77_NORMALIZADA.csv')
    df_completo.to_csv(ruta_limpieza, index=False)

    print(f"✅ ¡Normalización terminada! Archivo listo para el 'Matching' de fotos.")
    display(data_table.DataTable(df_completo[['Codigo', 'Titulo_Comercial', 'Medida_Final', 'Venta_Sugerida_65', 'Categoria', 'Linea']].head(20), include_index=False))
else:
    print("⚠️ DataFrame de productos vacío, no se realizó la normalización.")

## 📸 Vinculador Multi-Foto: Gestión de Galerías de Imágenes

Esta celda se encarga de vincular hasta 9 fotos por producto, buscando las imágenes en una estructura de carpetas definida y asociándolas a cada registro del DataFrame. Los pasos clave son:

*   **Funciones de Utilidad:** Incluye funciones como `limpieza_extrema` y `clean_string_for_output_robust` para estandarizar los nombres de las carpetas y archivos, facilitando la coincidencia.
*   **Ruta Base de Fotos:** Define `PATH_FOTOS_BASE` para especificar dónde se encuentran las imágenes de los productos. Esta ruta ha sido corregida para ser relativa a `BASE_PATH`, cumpliendo con el estándar de portabilidad.
*   **Carga de Datos Normalizados:** Carga el archivo `LISTA_77_NORMALIZADA.csv`, que contiene los datos de productos limpios y sus títulos comerciales.
*   **Mapeo de Carpetas:** Crea un mapeo de nombres de carpetas de categorías de fotos (limpias) a sus nombres reales, lo que permite buscar imágenes de manera flexible.
*   **`obtener_galeria_fotos`:** Esta función busca imágenes dentro de una carpeta específica para un modelo o categoría, priorizando archivos con extensiones de imagen comunes o nombres numéricos.
*   **`vincular_galeria`:** Esta es la función principal que, para cada producto:
    *   Extrae el modelo del código del producto.
    *   Normaliza el nombre de la categoría para encontrar la carpeta de fotos correspondiente.
    *   Busca fotos primero en una subcarpeta específica del modelo y, si no encuentra, busca en carpetas genéricas dentro de la categoría.
*   **Asignación de Galerías:** Añade una nueva columna `Fotos_Galeria` al DataFrame, que contiene una cadena con las rutas de las imágenes encontradas para cada producto.
*   **Métricas y Guardado:** Calcula el promedio de fotos por producto y el total de productos con galería. Finalmente, guarda el DataFrame actualizado con las galerías de fotos en `LISTA_77_CON_GALERIAS.csv`.

In [ ]:
# @title 📸 VINCULADOR MULTI-FOTO: Hasta 9 fotos por producto
import os
import pandas as pd
import unicodedata
from tqdm.auto import tqdm
import re # Importar re para usar expresiones regulares

tqdm.pandas() # Habilitar tqdm para pandas

# --- Funciones de Utilidad (copiadas de celdas anteriores para autosuficiencia) ---
def limpieza_extrema(texto):
    if not texto: return ""
    texto = unicodedata.normalize('NFD', str(texto))
    texto = "".join([c for c in texto if unicodedata.category(c).startswith('L')])
    return texto.upper().strip()

def clean_string_for_output_robust(s):
    if not isinstance(s, str):
        return str(s)
    try:
        return s.encode('utf-8', errors='replace').decode('utf-8', errors='replace')
    except Exception:
        return s.encode('ascii', errors='replace').decode('ascii', errors='replace')
# ---------------------------------------------------------------------------------

# Se modificó PATH_FOTOS_BASE para que sea relativa a BASE_PATH y no una ruta absoluta.
# Asume que 'ARTICULOS LOUISIANA/ART. LOUISIANA' está dentro de la carpeta BASE_PATH.
# Si 'ARTICULOS LOUISIANA' es una carpeta hermana o en la raíz de Drive, ajusta esta ruta.
PATH_FOTOS_BASE = os.path.join(BASE_PATH.split('/MyDrive/')[0], 'MyDrive', 'ARTICULOS LOUISIANA', 'ART. LOUISIANA')

ruta_csv = os.path.join(BASE_PATH, 'procesados', 'LISTA_77_NORMALIZADA.csv') # Cargar el CSV con la normalización de descripciones
df = pd.read_csv(ruta_csv)

# Comentarios y prints de DEBUG eliminados para mayor profesionalismo
# print(f"DEBUG: Checking PATH_FOTOS_BASE: {PATH_FOTOS_BASE}")
# print(f"DEBUG: os.path.exists(PATH_FOTOS_BASE): {os.path.exists(PATH_FOTOS_BASE)}")

# Crear mapa_carpetas una vez, fuera de la función, para que sea accesible
mapa_carpetas = {}
if os.path.exists(PATH_FOTOS_BASE):
    # print(f"DEBUG: Listing contents of {PATH_FOTOS_BASE}...") # Debug print removed
    for d_raw in os.listdir(PATH_FOTOS_BASE):
        full_path_d = os.path.join(PATH_FOTOS_BASE, d_raw)
        is_dir = os.path.isdir(full_path_d)
        # print(f"DEBUG: Item: {d_raw}, is_dir: {is_dir}") # Debug print removed
        try:
            if is_dir:
                nombre_limpio_d = limpieza_extrema(d_raw)
                mapa_carpetas[nombre_limpio_d] = d_raw
                # print(f"DEBUG: Added to mapa_carpetas: {nombre_limpio_d} -> {d_raw}") # Debug print removed
        except Exception as e:
            print(f"⚠️ Error al procesar nombre de carpeta '{clean_string_for_output_robust(d_raw)}': {clean_string_for_output_robust(str(e))}")
    # print(f"DEBUG: Final mapa_carpetas: {mapa_carpetas}") # Debug print removed

def obtener_galeria_fotos(ruta_objetivo, max_fotos=9):
    """Busca hasta N imágenes dentro de una ruta y las devuelve como una lista.
       Considera archivos con y sin extensión si son numéricos."""
    fotos_encontradas = []
    try:
        if os.path.exists(ruta_objetivo) and os.path.isdir(ruta_objetivo):
            archivos = sorted(os.listdir(ruta_objetivo)) # sorted para que mantenga un orden
            for f_raw in archivos:
                full_file_path = os.path.join(ruta_objetivo, f_raw)
                if os.path.isfile(full_file_path):
                    f_cleaned = clean_string_for_output_robust(f_raw)
                    # Verificar si termina con extensiones de imagen comunes O si no tiene extensión y es numérico
                    if f_cleaned.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')) or \
                       (os.path.splitext(f_cleaned)[1] == '' and re.match(r'^\d+$', f_cleaned)): # Regex corregida: r'^\d+$' -> r'^\d+$'
                        fotos_encontradas.append(clean_string_for_output_robust(full_file_path))
                    if len(fotos_encontradas) >= max_fotos:
                        break
        return fotos_encontradas
    except Exception as e:
        # print(f"Error en obtener_galeria_fotos para {ruta_objetivo}: {e}") # Descomentar para depuración
        return []

def vincular_galeria(row):
    # 1. Extraer modelo
    codigo_full = str(row['Codigo'])
    modelo = codigo_full.split('/')[0].strip()

    # 2. Determinar la carpeta de la categoría (normalizada para el match)
    cat_producto_limpia = limpieza_extrema(row['Categoria'])

    nombre_carpeta_real = None
    for limpio, real in mapa_carpetas.items():
        if cat_producto_limpia in limpio or limpio in cat_producto_limpia:
            nombre_carpeta_real = real
            break

    if not nombre_carpeta_real:
        return "SIN FOTO"

    ruta_cat = os.path.join(PATH_FOTOS_BASE, nombre_carpeta_real)

    if os.path.exists(ruta_cat):
        # Intento A: Fotos del modelo específico (ej: /SÁBANAS/8540)
        ruta_modelo = os.path.join(ruta_cat, modelo)
        fotos = obtener_galeria_fotos(ruta_modelo)
        if fotos: return ", ".join(fotos)

        # Intento B: Si no hay carpeta de modelo, o está vacía, buscar una genérica en subcarpetas de la categoría
        for item in os.listdir(ruta_cat):
            ruta_item = os.path.join(ruta_cat, item)
            if os.path.isdir(ruta_item):
                fotos_emergencia = obtener_galeria_fotos(ruta_item)
                if fotos_emergencia: return ", ".join(fotos_emergencia)
            # Si el item es directamente un archivo de imagen en la raíz de la categoría (como '8532' en 'SÁBANAS')
            elif os.path.isfile(ruta_item):
                 f_cleaned = clean_string_for_output_robust(item)
                 if f_cleaned.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')) or \
                    (os.path.splitext(f_cleaned)[1] == '' and re.match(r'^\d+$', f_cleaned)): # Regex corregida
                     return clean_string_for_output_robust(os.path.join(ruta_cat, item))

    return "SIN FOTO"

print("🚀 Generando galerías de hasta 9 fotos por producto...")
df['Fotos_Galeria'] = df.progress_apply(vincular_galeria, axis=1)

# Contabilizar cuántas fotos promedio conseguimos
df['Cant_Fotos'] = df['Fotos_Galeria'].apply(lambda x: len(x.split(',')) if x != "SIN FOTO" else 0)
promedio = df['Cant_Fotos'].mean()

print(f"✅ ¡Proceso terminado!")
print(f"📸 Promedio de fotos por producto: {promedio:.1f}")
print(f"📦 Total productos con galería: {len(df[df['Cant_Fotos'] > 0])}")

# Guardar versión final con galerías
ruta_galeria = os.path.join(BASE_PATH, 'resultados', 'LISTA_77_CON_GALERIAS.csv')
df.to_csv(ruta_galeria, index=False)

## ⚖️ GENERADOR TIENDANUBE: Con Pesos Reales, SKUs Únicos y Descripciones Optimizadas

Esta celda finaliza la preparación de los datos para su exportación a plataformas como Tiendanube. Se encarga de:

*   **Asignación de Pesos:** Utiliza una `tabla_pesos` predefinida para asignar un peso (en kg) a cada producto basándose en su `Medida_Final`, lo cual es crucial para la configuración de envíos.
*   **Generación de SKUs Únicos:** Crea identificadores únicos (`SKU`) combinando el código del producto y su medida. Incluye una lógica para manejar y resolver posibles duplicados añadiendo el índice del DataFrame.
*   **Construcción de Descripciones Optimizadas:** Genera descripciones de productos ricas en detalles y personalizadas, adaptándose a la calidad del material (ej. microfibra de 70, 90, 120 grs) y fomentando la compra.
*   **Estructura de Exportación:** Organiza todos los datos (nombre, categoría, precio, stock, fotos, etc.) en una estructura de DataFrame compatible con los requisitos de importación de Tiendanube.
*   **Exportación y Descarga:** Guarda el DataFrame final en un archivo CSV con codificación `utf-8-sig` (necesario para Excel y compatibilidad con Tiendanube) y automáticamente ofrece la descarga del archivo.

In [ ]:
# @title ⚖️ GENERADOR TIENDANUBE: Con Pesos Reales, SKUs Únicos y Descripciones Optimizadas
import pandas as pd
import os
from google.colab import files

# 1. Definimos la tabla de pesos (en kg)
tabla_pesos = {
    '1 1/2 PLAZA': 1.2,
    'TWIN': 1.2,
    '2 1/2 PLAZAS': 1.6,
    'QUEEN SIZE': 1.8,
    'KING SIZE': 2.1,
    'CUNA': 0.5,
    'CUNA FUNCIONAL': 0.6,
    'COLECHO': 0.4,
    'ALMOHADA 70X40': 0.3,
    'ESTÁNDAR': 0.8 # Peso genérico para productos sin medida específica
}

# 2. Cargamos la base procesada con galerías de fotos
ruta_galeria = os.path.join(BASE_PATH, 'resultados', 'LISTA_77_CON_GALERIAS.csv')
df = pd.read_csv(ruta_galeria)

def asignar_peso(medida):
    medida_upper = str(medida).upper().replace(' ', '') # Normalizar medida
    for clave, peso in tabla_pesos.items():
        if clave.replace(' ', '') in medida_upper: # Comparar también normalizado
            return peso
    return tabla_pesos['ESTÁNDAR'] # Asignar genérico si no hay match

def generar_sku_limpio(row):
    # Tomamos el código (ej: 08520/1) y la medida (ej: 1 1/2 PZA)
    codigo = str(row['Codigo']).replace('/', '-').strip()
    medida = str(row['Medida_Final']).replace(' ', '').upper()
    return f"{codigo}-{medida}"

def construir_descripcion(row):
    titulo = str(row['Titulo_Comercial'])
    categoria = str(row['Categoria']).capitalize()

    # Detectamos calidad para personalizar el texto
    calidad = ""
    if "70 GRS" in titulo.upper():
        calidad = " de microfibra de 70 grs, ideal por su suavidad y secado rápido."
    elif "90 GRS" in titulo.upper():
        calidad = " de microfibra premium de 90 grs, con un cuerpo y textura superior."
    elif "120 GRS" in titulo.upper():
        calidad = " de alta gama (120 grs), máxima suavidad y durabilidad."
    else:
        calidad = " confeccionada con materiales seleccionados para garantizar tu descanso."

    cuerpo = (
        f"¡Renová tu hogar con Creaciones Infinitas!\n\n"
        f"Este producto es una {categoria} {calidad}\n\n"
        f"✅ Producto de fabricación nacional.\n"
        f"✅ Terminaciones reforzadas para mayor durabilidad.\n"
        f"✅ Diseño exclusivo de nuestra línea de productos.\n\n" # Edited this line
        f"En Creaciones Infinitas nos apasiona cuidar cada detalle de tu dormitorio. "
        f"¡Sumalo a tu carrito y disfrutá del confort que te merecés!"
    )
    return cuerpo

# 3. Creamos el archivo con la estructura final
df_nube = pd.DataFrame()

df_nube['Nombre'] = df['Titulo_Comercial']
df_nube['Categoría'] = df['Categoria']
df_nube['Nombre de propiedad 1'] = 'Medida'
df_nube['Valor de propiedad 1'] = df['Medida_Final']
df_nube['Precio'] = df['Venta_Sugerida_65']
df_nube['Precio promocional'] = ''
df_nube['Peso (kg)'] = df['Medida_Final'].apply(asignar_peso) # <--- PESO REAL APLICADO
df_nube['Stock'] = '999'

# Generamos SKUs y manejamos duplicados
df_nube['SKU'] = df.apply(generar_sku_limpio, axis=1)
duplicados_sku = df_nube['SKU'].duplicated().sum()
if duplicados_sku > 0:
    print(f"⚠️ Atención: Se encontraron {duplicados_sku} SKUs duplicados. Ajustando...")
    df_nube['SKU'] = df_nube['SKU'] + "-" + df_nube.index.astype(str)

df_nube['Código de barras'] = df['Codigo'] # Se mantiene el código original aquí
df_nube['Mostrar en tienda'] = 'SI'
df_nube['Envío sin cargo'] = 'NO'
df_nube['Descripción'] = df.apply(construir_descripcion, axis=1)

# Repartimos las fotos en columnas Imagen 1, Imagen 2...
fotos_split = df['Fotos_Galeria'].str.split(', ', expand=True)
for i in range(min(9, fotos_split.shape[1])):
    df_nube[f'Imagen {i+1}'] = fotos_split[i]

# 4. Guardar y Descargar
ruta_tiendanube_final = os.path.join(BASE_PATH, 'resultados', 'TIENDANUBE_LISTA_77_FINAL.csv')
df_nube.to_csv(ruta_tiendanube_final, index=False, encoding='utf-8-sig')

print("✅ Archivo final para Tiendanube generado con éxito!")
print(f"📍 Ubicación: {ruta_tiendanube_final}")
files.download(ruta_tiendanube_final)

display(data_table.DataTable(df_nube.head(10), include_index=False))

### ✨ Validación de Distribución de Categorías

Esta celda verifica la cantidad de productos por cada categoría presente en el DataFrame final (`df_nube`), asegurando una distribución lógica y esperada de los productos listos para exportar.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Contar la distribución de categorías
category_distribution = df_nube['Categoría'].value_counts().reset_index()
category_distribution.columns = ['Categoría', 'Cantidad_Productos']

print("📊 Distribución de Categorías en el DataFrame final:")
display(data_table.DataTable(category_distribution, include_index=False))

# Opcional: Visualizar la distribución de categorías
fig = plt.figure(figsize=(10, 6))
# Se corrige la advertencia de FutureWarning asignando 'hue' y 'legend=False'
sns.barplot(x='Categoría', y='Cantidad_Productos', data=category_distribution, palette='viridis', hue='Categoría', legend=False)
plt.title('Distribución de Productos por Categoría')
plt.xlabel('Categoría')
plt.ylabel('Cantidad de Productos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()